# Saddle Points — Section 4.2

A **saddle point** is a stationary point (gradient = 0) that is neither a minimum nor a maximum.

The canonical example is $f(x, y) = x^2 - y^2$:
- $\partial f / \partial x = 2x$ → positive curvature in $x$
- $\partial f / \partial y = -2y$ → negative curvature in $y$

At $(0,0)$: $\nabla f = 0$ exactly. Gradient descent has **no gradient signal** and cannot escape — regardless of learning rate.

**Momentum** accumulates velocity from past gradients and can carry the optimizer through zero-gradient regions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def f(x, y):   return x**2 - y**2
def gx(x, y):  return 2 * x
def gy(x, y):  return -2 * y


def run_gd(x0, y0, lr, steps, mode='pure_gd', beta=0.85, noise_scale=0.02, seed=0):
    rng = np.random.default_rng(seed)
    path = [(x0, y0)]
    vx, vy = 0.0, 0.0
    for _ in range(steps):
        x, y = path[-1]
        noise = rng.normal(0, noise_scale, 2) if mode == 'perturbed' else np.zeros(2)
        if mode == 'momentum':
            vx = beta * vx - lr * gx(x, y)
            vy = beta * vy - lr * gy(x, y)
            nx, ny = x + vx, y + vy
        else:
            nx = x - lr * gx(x, y) + noise[0]
            ny = y - lr * gy(x, y) + noise[1]
        path.append((np.clip(nx, -3, 3), np.clip(ny, -3, 3)))
    return np.array(path)


def draw_saddle(x0=0.5, y0=0.3, lr=0.1, steps=60, mode='pure_gd'):
    path = run_gd(x0, y0, lr, steps, mode)

    xs = np.linspace(-2.5, 2.5, 60)
    ys = np.linspace(-2.5, 2.5, 60)
    XX, YY = np.meshgrid(xs, ys)
    ZZ = f(XX, YY)

    fig = plt.figure(figsize=(14, 5))

    # 3D surface
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(XX, YY, ZZ, cmap='RdBu_r', alpha=0.7, linewidth=0)
    ax1.plot(path[:, 0], path[:, 1], f(path[:, 0], path[:, 1]) + 0.05,
             '-o', color='#2563eb', ms=3, lw=2, label='GD path')
    ax1.scatter([0], [0], [0.05], c='#dc2626', s=80, marker='x', zorder=5, label='Saddle (0,0)')
    ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('f(x,y)')
    ax1.set_title('f(x,y) = x² − y²  (3D)', fontsize=11)
    ax1.legend(fontsize=8)

    # 2D contour + trajectory
    ax2 = fig.add_subplot(122)
    ax2.contourf(XX, YY, ZZ, levels=20, cmap='RdBu_r', alpha=0.5)
    ax2.contour(XX, YY, ZZ, levels=20, colors='gray', linewidths=0.5, alpha=0.4)

    # Gradient arrows
    qx = np.linspace(-2, 2, 10)
    QX, QY = np.meshgrid(qx, qx)
    U, V = -gx(QX, QY), -gy(QX, QY)
    mag = np.sqrt(U**2 + V**2) + 1e-8
    ax2.quiver(QX, QY, U/mag, V/mag, alpha=0.25, scale=25, color='#475569')

    ax2.plot(path[:, 0], path[:, 1], '-o', color='#2563eb', ms=4, lw=2, label='GD path')
    ax2.scatter([0], [0], c='#dc2626', s=100, marker='x', zorder=5, linewidths=2, label='Saddle')
    ax2.scatter([x0], [y0], c='#059669', s=80, marker='*', zorder=5, label='Start')
    ax2.set_xlim(-2.5, 2.5); ax2.set_ylim(-2.5, 2.5)
    ax2.set_xlabel('x'); ax2.set_ylabel('y')
    ax2.set_title('Contour + GD trajectory', fontsize=11)
    ax2.legend(fontsize=8)

    mode_labels = {'pure_gd': 'Pure GD (locks at saddle)', 'perturbed': 'GD + noise', 'momentum': 'GD + momentum (escapes!)'}
    plt.suptitle(f'{mode_labels[mode]}   α={lr}   steps={steps}', fontsize=10, y=0.0)
    plt.tight_layout()
    plt.show()

    last = path[-1]
    grad_mag = np.sqrt(gx(*last)**2 + gy(*last)**2)
    print(f'Final position: x={last[0]:.4f}, y={last[1]:.4f}')
    print(f'f(x,y) = {f(*last):.6f}   |∇f| = {grad_mag:.6f}')
    if grad_mag < 1e-4 and mode == 'pure_gd':
        print('⚠ Stuck at saddle — gradient ≈ 0. No learning rate will fix this!')
    elif mode == 'momentum' and abs(last[1]) > 0.3:
        print('✓ Escaped! Momentum carried the optimizer past the zero-gradient saddle point.')


widgets.interact(
    draw_saddle,
    x0=widgets.FloatSlider(min=-1.8, max=1.8, step=0.1, value=0.5, description='Start x₀', continuous_update=False),
    y0=widgets.FloatSlider(min=-1.8, max=1.8, step=0.1, value=0.3, description='Start y₀', continuous_update=False),
    lr=widgets.SelectionSlider(options=[0.001, 0.01, 0.05, 0.1, 0.3, 0.5], value=0.1, description='Learning rate α', continuous_update=False),
    steps=widgets.IntSlider(min=10, max=200, step=10, value=60, description='Steps', continuous_update=False),
    mode=widgets.Dropdown(options=[('Pure GD (locks at saddle)', 'pure_gd'), ('GD + tiny noise', 'perturbed'), ('GD + momentum (escapes!)', 'momentum')], value='pure_gd', description='Mode'),
)